In [5]:
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass
from typing import Optional
import json
import csv
import re


@dataclass
class Festival:
    nom: str
    dates: str
    lieu: str
    artistes: list[str]
    billetterie_url: Optional[str] = None


def scrape_festivals(url: str) -> list[Festival]:
    """
    Scrape les informations des festivals depuis la page web.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, 'html.parser')
    festivals = []
    
    # Trouver tous les h3 (noms des festivals)
    h3_tags = soup.find_all('h3')
    
    for h3 in h3_tags:
        nom_complet = h3.get_text(strip=True)
        
        # Extraire le nom et les dates
        match = re.match(r'^(.+?)\s*:\s*(.+)$', nom_complet)
        if match:
            nom = match.group(1).strip()
            dates = match.group(2).strip()
        else:
            continue
        
        # Trouver le contenu suivant le h3
        contenu = []
        sibling = h3.find_next_sibling()
        
        lieu = ""
        artistes = []
        billetterie = None
        
        while sibling and sibling.name != 'h3' and sibling.name != 'h2':
            text = sibling.get_text(strip=True)
            
            # Chercher le lieu (ligne commençant par les dates)
            if text.startswith('Du ') or text.startswith('Le '):
                lieu_match = re.search(r'(?:à|au|dans)\s+(.+?)(?:\s*La billetterie|$)', text)
                if lieu_match:
                    lieu = lieu_match.group(1).strip()
            
            # Chercher les liens de billetterie
            links = sibling.find_all('a', href=True)
            for link in links:
                href = link['href']
                if 'billetterie' in link.get_text().lower() or any(
                    domain in href for domain in [
                        'welovegreen', 'solidays', 'rockenseine', 'hellfest',
                        'eurockeennes', 'francofolies', 'vieillescharrues',
                        'garorock', 'musilac', 'mainsquare', 'cabaretvert'
                    ]
                ):
                    billetterie = href if href.startswith('http') else None
            
            # Chercher les artistes (liens vers /artiste/)
            artiste_links = sibling.find_all('a', href=re.compile(r'/artiste/'))
            for link in artiste_links:
                artiste = link.get_text(strip=True)
                if artiste and artiste not in artistes:
                    artistes.append(artiste)
            
            sibling = sibling.find_next_sibling()
        
        if nom and dates:
            festivals.append(Festival(
                nom=nom,
                dates=dates,
                lieu=lieu,
                artistes=artistes,
                billetterie_url=billetterie
            ))
    
    return festivals

def export_to_json(festivals: list[Festival], filename: str = "festivals_2026.json"):
    """Exporte les festivals en JSON."""
    data = [
        {
            "nom": f.nom,
            "dates": f.dates,
            "lieu": f.lieu,
            "artistes": f.artistes,
            "billetterie": f.billetterie_url
        }
        for f in festivals
    ]
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"✅ Exporté vers {filename}")

def print_festivals(festivals: list[Festival]):
    """Affiche les festivals de manière formatée."""
    print("\n" + "=" * 80)
    print("🎵 FESTIVALS MUSICAUX DE L'ÉTÉ 2026 🎵")
    print("=" * 80)
    
    for fest in festivals:
        print(f"\n🎪 {fest.nom}")
        print(f"   📅 {fest.dates}")
        print(f"   📍 {fest.lieu}")
        print(f"   🎤 {', '.join(fest.artistes[:5])}", end="")
        if len(fest.artistes) > 5:
            print(f" (+{len(fest.artistes) - 5} autres)")
        else:
            print()
        if fest.billetterie_url:
            print(f"   🎟️  {fest.billetterie_url}")
        print("-" * 60)

if __name__ == "__main__":
    # Récupérer les données
    festivals = scrape_festivals("https://www.offi.fr/tendances/concerts/les-grands-festivals-musicaux-de-lete-2026-942.html")
    # Afficher les festivals
    print_festivals(festivals)
    # Exporter en JSON et CSV
    export_to_json(festivals)



🎵 FESTIVALS MUSICAUX DE L'ÉTÉ 2026 🎵

🎪 We Love Green
   📅 du 5 au 7 juin 2026
   📍 7 juin 2026 dans le bois de Vincennes
   🎤 Gorillaz, Theodora, The xx, Charlotte Cardin, Addison Rae (+7 autres)
   🎟️  https://www.welovegreen.fr/
------------------------------------------------------------

🎪 Festival des 2 Rivières
   📅 du 12 au 14 juin 2026
   📍 14 juin 2026 à La Ferté-sous-Jouarre (77)
   🎤 Superbus, Fatoumata Diawara, Gaël Faye, Claudio Capéo
------------------------------------------------------------

🎪 Solidays
   📅 du 26 au 28 juin 2026
   📍 28 juin 2026 à l'Hippodrome de Longchamp
   🎤 Orelsan, Gims, Bigflo & Oli, L2B, Luiza (+4 autres)
   🎟️  https://www.solidays.org/
------------------------------------------------------------

🎪 Rock en Seine
   📅 du 26 au 30 août 2026
   📍 30 août 2026 dans le domaine de Saint-Cloud
   🎤 The Cure, Nick Cave & The Bad Seeds, Tyler, The Creator, Deftones, The Black Keys (+11 autres)
   🎟️  https://www.rockenseine.com
---------------------